# EcoCor Entity Co-occurrence Networks v2

This notebook:
1. **Step 1** – Fetches Animal and Plant entities from the EcoCor API (with local cache)
2. **Step 2** – Builds a co-occurrence network for each text in the corpus
3. **Step 3** – Merges all individual networks into one corpus-level network via Wikidata Q-IDs
4. **Step 4** – Generates interactive Bokeh plots for individual text and corpus networks

## Section 0 – Configuration & Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import time
from pathlib import Path
from itertools import combinations

# ── Data ──────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Network ───────────────────────────────────────────────────────────────────
import networkx as nx
from networkx.algorithms import community as nx_community

# ── HTTP ──────────────────────────────────────────────────────────────────────
import requests

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.cm as mpl_cm
import matplotlib.colors as mpl_colors

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import (
    ColumnDataSource, HoverTool, MultiLine, Circle,
    Slider, Select, CustomJS, TapTool,
    WheelZoomTool, BoxZoomTool, PanTool, ResetTool,
)
from bokeh.palettes import Category20
from bokeh.layouts import column, row

output_notebook()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# USER-CONFIGURABLE PARAMETERS  –  adjust as needed
# ═══════════════════════════════════════════════════════════════════════════════

ECOCOR_API_BASE = "https://ecocor.org/api"
TARGET_CORPUS   = ""                      # corpus name: "en", "de" …
WINDOW_SIZE     = 1                       # co-occurrence window (paragraphs)
ENTITY_TYPES    = ["Animal", "Plant"]     # entity types to fetch
MIN_EDGE_WEIGHT = 3                       # default filter value for plots

# Paths
BASE_DIR  = Path("")
CACHE_DIR = BASE_DIR / "cache_en"
OUT_DIR   = BASE_DIR / "output_en"

# ── Create output directories ─────────────────────────────────────────────────
(CACHE_DIR / TARGET_CORPUS).mkdir(parents=True, exist_ok=True)
(OUT_DIR / "per_text").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "corpus").mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Corpus       : {TARGET_CORPUS}")
print(f"  Window size  : {WINDOW_SIZE} paragraph(s)")
print(f"  Entity types : {ENTITY_TYPES}")
print(f"  Cache        : {CACHE_DIR}")
print(f"  Output       : {OUT_DIR}")

## Section 1 – Helper Functions

In [ ]:
# ── 1-A: Parse occurrence ID ──────────────────────────────────────────────────

def parse_occurrence_id(occ_id: str):
    """
    'eco_en_000026_10340' -> ('eco_en_000026', '10340', 1034)
    The last character of paragraph_code is a disambiguation suffix and is
    stripped when computing paragraph_index.
    """
    parts = occ_id.split("_")
    if len(parts) < 4:
        return None, None, None
    text_id         = "_".join(parts[:3])
    paragraph_code  = parts[3]
    paragraph_index = int(paragraph_code[:-1]) if len(paragraph_code) > 1 else int(paragraph_code)
    return text_id, paragraph_code, paragraph_index

In [ ]:
# ── 1-B: Add window index ─────────────────────────────────────────────────────

def add_window_index(df: pd.DataFrame, window_size: int = 1) -> pd.DataFrame:
    """
    Adds a 'window_index' column to *df*.
    window_size=1: one window per paragraph.
    window_size=N: non-overlapping windows of N consecutive paragraphs.
    """
    if window_size < 1:
        raise ValueError("window_size must be >= 1")
    df = df.copy()
    if window_size == 1:
        df["window_index"] = df["paragraph_index"]
    else:
        df["window_index"] = ((df["paragraph_index"] - 1) // window_size) + 1
    df["window_size"] = window_size
    return df

In [ ]:
# ── 1-C: Entities JSON → flat DataFrame ───────────────────────────────────────

def entities_to_df(entities: list) -> pd.DataFrame:
    """
    Converts the API JSON list (Animal + Plant combined) into a flat
    occurrence DataFrame.

    Columns: entity_qid, entity_name, entity_type, overall_frequency,
             occurrence_id, text_id, paragraph_code, paragraph_index,
             occurrence_frequency
    """
    rows = []
    for ent in entities:
        qid     = ent.get("id")
        name    = ent.get("name")
        etype   = ent.get("type")
        metrics = ent.get("metrics", {})
        overall = metrics.get("overallFrequency")
        for occ in metrics.get("occurrences", []):
            occ_id = occ.get("id")
            freq   = occ.get("frequency", 1)
            text_id, paragraph_code, paragraph_index = parse_occurrence_id(occ_id)
            rows.append({
                "entity_qid":           qid,
                "entity_name":          name,
                "entity_type":          etype,
                "overall_frequency":    overall,
                "occurrence_id":        occ_id,
                "text_id":              text_id,
                "paragraph_code":       paragraph_code,
                "paragraph_index":      paragraph_index,
                "occurrence_frequency": freq,
            })
    return pd.DataFrame(rows)

In [ ]:
# ── 1-D: Build network from occurrence DataFrame ──────────────────────────────

def build_network_from_df(df_occ: pd.DataFrame, window_size: int = 1):
    """
    Builds a nodes DataFrame, an edges DataFrame, and a NetworkX graph from
    an occurrence DataFrame.  Nodes are identified by Wikidata Q-IDs.
    Returns: (nodes_df, edges_df, G)
    """
    if df_occ.empty:
        G = nx.Graph()
        return pd.DataFrame(), pd.DataFrame(columns=["source","target","weight"]), G

    df_win = add_window_index(df_occ, window_size=window_size)

    # Nodes
    nodes_df = (
        df_win[["entity_qid", "entity_name", "entity_type", "overall_frequency"]]
        .drop_duplicates(subset=["entity_qid"])
        .reset_index(drop=True)
    )

    # Edges: all pairs of Q-IDs co-occurring within the same window
    edge_rows = []
    for (text_id, window_idx), group in df_win.groupby(["text_id", "window_index"]):
        qids = group["entity_qid"].dropna().unique().tolist()
        if len(qids) < 2:
            continue
        for src_qid, tgt_qid in combinations(sorted(qids), 2):
            edge_rows.append({
                "source":              src_qid,
                "target":              tgt_qid,
                "context_text_id":     text_id,
                "context_window_idx":  window_idx,
                "weight":              1,
            })

    if not edge_rows:
        edges_df = pd.DataFrame(columns=["source","target","weight"])
    else:
        edges_df = (
            pd.DataFrame(edge_rows)
            .groupby(["source", "target"], as_index=False)["weight"]
            .sum()
        )

    # NetworkX graph
    G = nx.Graph()
    for _, r in nodes_df.iterrows():
        G.add_node(r["entity_qid"],
                   name              = r["entity_name"],
                   entity_type       = r["entity_type"],
                   overall_frequency = int(r["overall_frequency"] or 0))
    for _, r in edges_df.iterrows():
        G.add_edge(r["source"], r["target"], weight=float(r["weight"]))

    return nodes_df, edges_df, G

In [ ]:
# ── 1-E: Compute network metrics ──────────────────────────────────────────────

def compute_network_metrics(G: nx.Graph) -> pd.DataFrame:
    """
    Computes per-node metrics:
      degree, weighted_degree, betweenness_centrality,
      clustering_coefficient, community_id
    Also writes 'community' back as a node attribute on G.
    Returns: DataFrame
    """
    if G.number_of_nodes() == 0:
        return pd.DataFrame()

    degree       = dict(G.degree())
    weighted_deg = dict(G.degree(weight="weight"))

    if G.number_of_edges() > 0:
        betweenness = nx.betweenness_centrality(G, weight="weight", normalized=True)
        clustering  = nx.clustering(G, weight="weight")
        communities = list(nx_community.greedy_modularity_communities(G, weight="weight"))
    else:
        betweenness = {n: 0.0 for n in G.nodes()}
        clustering  = {n: 0.0 for n in G.nodes()}
        communities = [frozenset(G.nodes())]

    comm_map = {}
    for idx, comm in enumerate(communities):
        for node in comm:
            comm_map[node] = idx
    nx.set_node_attributes(G, comm_map, "community")

    rows = []
    for node in G.nodes():
        rows.append({
            "node_qid":               node,
            "degree":                 degree[node],
            "weighted_degree":        weighted_deg[node],
            "betweenness_centrality": round(betweenness[node], 6),
            "clustering_coefficient": round(clustering[node], 6),
            "community_id":           comm_map.get(node, -1),
        })
    return pd.DataFrame(rows)

## Section 2 – Step 1: Fetch Entity Data from the EcoCor API

In [ ]:
# ── 2-A: Query corpus texts ───────────────────────────────────────────────────

def get_corpus_texts(corpus_name: str) -> list:
    """
    Fetches all texts of a corpus from the EcoCor API.
    Returns: list of dicts with text_id, text_name, title.
    """
    url  = f"{ECOCOR_API_BASE}/corpora/{corpus_name}/texts"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    texts = resp.json()
    result = []
    for t in texts:
        result.append({
            "text_id":   t.get("id"),
            "text_name": t.get("name"),
            "title":     t.get("title", ""),
        })
    return result


corpus_texts = get_corpus_texts(TARGET_CORPUS)
print(f"Texts in corpus '{TARGET_CORPUS}': {len(corpus_texts)}")
for t in corpus_texts[:5]:
    print(f"  {t['text_id']}  |  {t['text_name']}  |  {t['title']}")
if len(corpus_texts) > 5:
    print(f"  … and {len(corpus_texts)-5} more")

In [ ]:
# ── 2-B: Fetch entities with caching ──────────────────────────────────────────

def fetch_entities_cached(corpus_name: str, text_name: str,
                           entity_type: str, cache_dir: Path,
                           force_refresh: bool = False) -> list:
    """
    Fetches entities for one text and type from the API.
    Results are cached as JSON. Returns an empty list on HTTP or network error.
    """
    corpus_cache = cache_dir / corpus_name
    corpus_cache.mkdir(parents=True, exist_ok=True)
    cache_file   = corpus_cache / f"{text_name}_{entity_type}.json"

    if cache_file.exists() and not force_refresh:
        with cache_file.open("r", encoding="utf-8") as f:
            return json.load(f)

    url = (f"{ECOCOR_API_BASE}/corpora/{corpus_name}"
           f"/texts/{text_name}/entities?type={entity_type}")
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()
    except requests.exceptions.HTTPError as e:
        print(f"    HTTP error {text_name}/{entity_type}: {e}")
        data = []
    except requests.exceptions.RequestException as e:
        print(f"    Network error {text_name}/{entity_type}: {e}")
        data = []

    with cache_file.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    time.sleep(0.3)   # polite rate-limiting
    return data


def fetch_all_entities(corpus_name: str, corpus_texts: list,
                        entity_types: list, cache_dir: Path,
                        force_refresh: bool = False) -> dict:
    """
    Fetches all entity types for every text and merges them into one list.
    Returns: { text_id: [merged entity list] }
    Texts with no entities are skipped.
    """
    all_entities = {}
    for t in corpus_texts:
        text_id   = t["text_id"]
        text_name = t["text_name"]
        merged    = []
        for etype in entity_types:
            entities = fetch_entities_cached(
                corpus_name, text_name, etype, cache_dir, force_refresh
            )
            merged.extend(entities)
        if merged:
            all_entities[text_id] = merged
            n_animal = sum(1 for e in merged if e.get("type") == "Animal")
            n_plant  = sum(1 for e in merged if e.get("type") == "Plant")
            print(f"  {text_id}: {len(merged)} entities "
                  f"({n_animal} Animal, {n_plant} Plant)")
        else:
            print(f"  {text_id}: no entities — skipped")
    return all_entities

In [ ]:
# ── 2-C: Start the fetch ──────────────────────────────────────────────────────
# Set force_refresh=True to bypass the cache and re-download all data.

print(f"Loading entities for corpus '{TARGET_CORPUS}' …")
all_entities = fetch_all_entities(
    corpus_name   = TARGET_CORPUS,
    corpus_texts  = corpus_texts,
    entity_types  = ENTITY_TYPES,
    cache_dir     = CACHE_DIR,
    force_refresh = False,
)
print(f"\nEntity data loaded for {len(all_entities)} texts.")

## Section 3 – Step 2: Build Individual Text Networks

In [ ]:
# ── 3-A: Build text-ID → metadata lookup ─────────────────────────────────────
text_meta_lookup = {t["text_id"]: t for t in corpus_texts}

# ── 3-B: Build all individual networks ───────────────────────────────────────
per_text_results = {}

for text_id, entities in all_entities.items():
    meta     = text_meta_lookup.get(text_id, {})
    df_occ   = entities_to_df(entities)

    nodes_df, edges_df, G = build_network_from_df(df_occ, window_size=WINDOW_SIZE)
    metrics_df            = compute_network_metrics(G)

    # Merge metrics into nodes_df
    if not metrics_df.empty:
        nodes_full = nodes_df.merge(
            metrics_df, left_on="entity_qid", right_on="node_qid", how="left"
        ).drop(columns=["node_qid"])
    else:
        nodes_full = nodes_df.copy()

    per_text_results[text_id] = {
        "df_occ":    df_occ,
        "nodes_df":  nodes_full,
        "edges_df":  edges_df,
        "G":         G,
        "text_meta": meta,
    }

    n_comms = len(list(nx_community.greedy_modularity_communities(G))) if G.number_of_edges() > 0 else 0
    print(f"{text_id}  [{meta.get('title', '')}]  "
          f"→ {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
          f"{n_comms} communities")

print(f"\nIndividual networks built: {len(per_text_results)}")

In [ ]:
# ── 3-C: Save per-text CSVs ───────────────────────────────────────────────────

for text_id, result in per_text_results.items():
    text_name = text_meta_lookup.get(text_id, {}).get("text_name", text_id)
    prefix    = f"{TARGET_CORPUS}_{text_name}"
    out       = OUT_DIR / "per_text"

    result["df_occ"].to_csv(
        out / f"{prefix}_occurrences.csv", index=False, encoding="utf-8")
    result["nodes_df"].to_csv(
        out / f"{prefix}_nodes.csv", index=False, encoding="utf-8")
    result["edges_df"].to_csv(
        out / f"{prefix}_edges_w{WINDOW_SIZE}.csv", index=False, encoding="utf-8")

print(f"Per-text CSVs saved to {OUT_DIR / 'per_text'}")

In [ ]:
# ── 3-D: Export per-text GEXF files (Gephi) ──────────────────────────────────
#
# GEXF is the recommended format for Gephi: it carries all node and edge
# attributes (name, entity_type, community, metrics) as typed fields that
# Gephi imports directly as Data Laboratory columns.
#
# Note: NetworkX writes GEXF 1.2draft, which Gephi reads without issues.
# All attributes must have uniform Python types — prepare_graph_for_gexf()
# ensures this.

def prepare_graph_for_gexf(G: nx.Graph) -> nx.Graph:
    """
    Returns a copy of G with all node and edge attributes normalised to
    plain Python types (str, int, float). None values are replaced with
    neutral defaults. Required for a clean GEXF export.
    """
    H = G.copy()
    for node, attrs in H.nodes(data=True):
        attrs["name"]                   = str(attrs.get("name") or node)
        attrs["entity_type"]            = str(attrs.get("entity_type") or "Unknown")
        attrs["overall_frequency"]      = int(attrs.get("overall_frequency") or 0)
        attrs["community"]              = int(attrs.get("community") or 0)
        attrs["degree"]                 = int(attrs.get("degree") or 0)
        attrs["weighted_degree"]        = float(attrs.get("weighted_degree") or 0.0)
        attrs["betweenness_centrality"] = float(attrs.get("betweenness_centrality") or 0.0)
        attrs["clustering_coefficient"] = float(attrs.get("clustering_coefficient") or 0.0)
        attrs["corpus_presence"]        = int(attrs.get("corpus_presence") or 0)
        attrs["texts_present"]          = str(attrs.get("texts_present") or "")
    for u, v, attrs in H.edges(data=True):
        attrs["weight"]        = float(attrs.get("weight") or 1.0)
        attrs["text_count"]    = int(attrs.get("text_count") or 0)
        attrs["texts_present"] = str(attrs.get("texts_present") or "")
    return H


out_gexf = OUT_DIR / "per_text"
for text_id, result in per_text_results.items():
    text_name = text_meta_lookup.get(text_id, {}).get("text_name", text_id)
    prefix    = f"{TARGET_CORPUS}_{text_name}"

    # Write metrics as node attributes into the graph copy
    G_exp    = result["G"].copy()
    nodes_df = result["nodes_df"]
    for col in ["degree", "weighted_degree", "betweenness_centrality",
                "clustering_coefficient", "community_id"]:
        if col in nodes_df.columns:
            attr_name = "community" if col == "community_id" else col
            mapping   = nodes_df.set_index("entity_qid")[col].to_dict()
            nx.set_node_attributes(G_exp, mapping, attr_name)

    G_clean = prepare_graph_for_gexf(G_exp)
    nx.write_gexf(G_clean, out_gexf / f"{prefix}_w{WINDOW_SIZE}.gexf")

print(f"Per-text GEXF files saved to {out_gexf}")

## Section 4 – Step 3: Corpus-Level Network (Merge)

In [ ]:
# ── 4-A: Merge function ───────────────────────────────────────────────────────

def build_corpus_network(per_text_results: dict):
    """
    Merges all individual text networks into one corpus-level network via
    Wikidata Q-IDs. Edge weights are accumulated across texts.
    Returns: (corpus_nodes_df, corpus_edges_df, corpus_provenance_df, G_corpus)
    """
    node_registry = {}   # qid → attrs
    edge_registry = {}   # (src_qid, tgt_qid) → attrs

    for text_id, result in per_text_results.items():
        G_text   = result["G"]
        edges_df = result["edges_df"]

        # Accumulate nodes
        for _, row in result["nodes_df"].iterrows():
            qid = row["entity_qid"]
            if qid not in node_registry:
                node_registry[qid] = {
                    "entity_qid":      qid,
                    "entity_name":     row["entity_name"],
                    "entity_type":     row["entity_type"],
                    "total_frequency": 0,
                    "texts_present":   set(),
                    "per_text_degree": {},
                }
            node_registry[qid]["total_frequency"] += int(row.get("overall_frequency") or 0)
            node_registry[qid]["texts_present"].add(text_id)
            node_registry[qid]["per_text_degree"][text_id] = int(
                row.get("degree", G_text.degree(qid) if qid in G_text else 0)
            )

        # Accumulate edges
        for _, row in edges_df.iterrows():
            src, tgt = sorted([row["source"], row["target"]])
            key = (src, tgt)
            if key not in edge_registry:
                edge_registry[key] = {"weight": 0, "texts_present": set()}
            edge_registry[key]["weight"]       += int(row["weight"])
            edge_registry[key]["texts_present"].add(text_id)

    # Node DataFrame
    node_rows = []
    for qid, attrs in node_registry.items():
        texts_list = sorted(attrs["texts_present"])
        node_rows.append({
            "entity_qid":      qid,
            "entity_name":     attrs["entity_name"],
            "entity_type":     attrs["entity_type"],
            "total_frequency": attrs["total_frequency"],
            "corpus_presence": len(attrs["texts_present"]),
            "texts_present":   ";".join(texts_list),
        })
    corpus_nodes_df = pd.DataFrame(node_rows)

    # Edge DataFrame
    edge_rows = []
    for (src, tgt), attrs in edge_registry.items():
        texts_list = sorted(attrs["texts_present"])
        edge_rows.append({
            "source":        src,
            "target":        tgt,
            "weight":        attrs["weight"],
            "texts_present": ";".join(texts_list),
            "text_count":    len(attrs["texts_present"]),
        })
    corpus_edges_df = pd.DataFrame(edge_rows) if edge_rows else pd.DataFrame(
        columns=["source", "target", "weight", "texts_present", "text_count"])

    # Provenance table (long format: qid × text_id)
    prov_rows = []
    for qid, attrs in node_registry.items():
        for text_id in attrs["texts_present"]:
            prov_rows.append({
                "entity_qid":     qid,
                "text_id":        text_id,
                "degree_in_text": attrs["per_text_degree"].get(text_id, 0),
            })
    corpus_provenance_df = pd.DataFrame(prov_rows)

    # NetworkX graph
    G_corpus = nx.Graph()
    for _, row in corpus_nodes_df.iterrows():
        G_corpus.add_node(
            row["entity_qid"],
            name            = row["entity_name"],
            entity_type     = row["entity_type"],
            total_frequency = row["total_frequency"],
            corpus_presence = row["corpus_presence"],
            texts_present   = row["texts_present"],
        )
    for _, row in corpus_edges_df.iterrows():
        G_corpus.add_edge(
            row["source"], row["target"],
            weight        = float(row["weight"]),
            text_count    = int(row["text_count"]),
            texts_present = row["texts_present"],
        )

    return corpus_nodes_df, corpus_edges_df, corpus_provenance_df, G_corpus

In [ ]:
# ── 4-B: Build the corpus network ────────────────────────────────────────────

corpus_nodes_df, corpus_edges_df, corpus_provenance_df, G_corpus = \
    build_corpus_network(per_text_results)

print(f"Corpus network: {G_corpus.number_of_nodes()} nodes, "
      f"{G_corpus.number_of_edges()} edges")

# Compute corpus-level metrics
corpus_metrics_df = compute_network_metrics(G_corpus)

# Merge metrics into corpus_nodes_df
if not corpus_metrics_df.empty:
    corpus_nodes_full = corpus_nodes_df.merge(
        corpus_metrics_df, left_on="entity_qid", right_on="node_qid", how="left"
    ).drop(columns=["node_qid"])
else:
    corpus_nodes_full = corpus_nodes_df.copy()

# Top 15 by degree
display(
    corpus_nodes_full[[
        "entity_qid", "entity_name", "entity_type",
        "corpus_presence", "degree", "betweenness_centrality", "community_id"
    ]].sort_values("degree", ascending=False).head(15)
)

In [ ]:
# ── 4-C: Save corpus CSVs ────────────────────────────────────────────────────

out = OUT_DIR / "corpus"
corpus_nodes_full.to_csv(
    out / f"{TARGET_CORPUS}_corpus_nodes.csv", index=False, encoding="utf-8")
corpus_edges_df.to_csv(
    out / f"{TARGET_CORPUS}_corpus_edges.csv", index=False, encoding="utf-8")
corpus_provenance_df.to_csv(
    out / f"{TARGET_CORPUS}_corpus_node_provenance.csv", index=False, encoding="utf-8")

print(f"Corpus CSVs saved to {out}")

In [ ]:
# ── 4-D: Export corpus-level GEXF file (Gephi) ───────────────────────────────
#
# The corpus network carries extra attributes not present in per-text files:
#   - corpus_presence (int):  number of texts the entity appears in
#   - texts_present   (str):  semicolon-separated list of text IDs
#   - text_count      (int):  on edges: number of texts sharing this edge
# In Gephi these allow colouring nodes by corpus presence, filtering edges
# by text_count, and sizing nodes by degree.

# Transfer metrics from corpus_nodes_full as node attributes
G_corpus_exp = G_corpus.copy()

for col in ["degree", "weighted_degree", "betweenness_centrality",
            "clustering_coefficient", "community_id",
            "corpus_presence", "total_frequency", "texts_present"]:
    if col in corpus_nodes_full.columns:
        attr_name = "community" if col == "community_id" else col
        mapping   = corpus_nodes_full.set_index("entity_qid")[col].to_dict()
        nx.set_node_attributes(G_corpus_exp, mapping, attr_name)

G_corpus_clean = prepare_graph_for_gexf(G_corpus_exp)

out_corpus_gexf = OUT_DIR / "corpus" / f"{TARGET_CORPUS}_corpus_w{WINDOW_SIZE}.gexf"
nx.write_gexf(G_corpus_clean, out_corpus_gexf)

print(f"Corpus GEXF saved: {out_corpus_gexf}")

## Section 5 – Step 4: Interactive Bokeh Visualisations

In [ ]:
# ── 5-A: Colour helper functions ──────────────────────────────────────────────

def community_color_map(n_communities: int, palette=None) -> dict:
    """Maps community_id (int) → hex colour string."""
    if palette is None:
        if n_communities <= 20:
            palette = Category20[max(n_communities, 3)]
        else:
            cmap    = mpl_cm.colormaps["tab20"].resampled(n_communities)
            palette = [mpl_colors.to_hex(cmap(i)) for i in range(n_communities)]
    return {i: palette[i % len(palette)] for i in range(n_communities)}


def presence_to_hex(corpus_presence: int, max_presence: int) -> str:
    """Maps corpus_presence → hex colour on a YlOrRd scale."""
    ratio = corpus_presence / max(max_presence, 1)
    return mpl_colors.to_hex(mpl_cm.colormaps["YlOrRd"](0.2 + 0.8 * ratio))

In [ ]:
# ── 5-B: Core graph renderer ──────────────────────────────────────────────────

def make_bokeh_graph(G: nx.Graph,
                     title: str = "EcoCor Entity Network",
                     min_edge_weight: float = 1.0,
                     width: int = 950,
                     height: int = 720):
    """
    Builds an interactive Bokeh plot for a NetworkX graph.

    Expected node attributes: name, entity_type, community (int),
      degree, weighted_degree, betweenness_centrality,
      clustering_coefficient, corpus_presence, texts_present
    Expected edge attributes: weight

    Returns: (figure, node_source, edge_source)
    """
    if G.number_of_nodes() == 0:
        print("Empty graph — nothing to plot.")
        return None, None, None

    # Filter edges by minimum weight
    G_f = nx.Graph()
    G_f.add_nodes_from(G.nodes(data=True))
    G_f.add_edges_from(
        (u, v, d) for u, v, d in G.edges(data=True)
        if d.get("weight", 1) >= min_edge_weight
    )

    # Layout
    n = G_f.number_of_nodes()
    k = 2.5 / max(n ** 0.5, 1)
    pos = nx.spring_layout(G_f, k=k, seed=42, iterations=120, weight="weight")

    # Community colours
    comm_ids = sorted(set(
        G_f.nodes[nd].get("community", 0) for nd in G_f.nodes()
    ))
    comm_colors      = community_color_map(len(comm_ids))
    comm_id_to_color = {c: comm_colors[i] for i, c in enumerate(comm_ids)}

    degrees    = dict(G_f.degree())
    max_degree = max(degrees.values()) if degrees else 1

    # Node data
    (node_ids, xs, ys, names, etypes, comms,
     colors, sizes, degs, wdegs, betw, clust,
     presence_list, texts_list) = ([] for _ in range(14))

    for nd in G_f.nodes():
        attrs = G_f.nodes[nd]
        x, y  = pos.get(nd, (0.0, 0.0))
        deg   = degrees.get(nd, 0)
        comm  = attrs.get("community", 0)

        node_ids.append(nd)
        xs.append(x);  ys.append(y)
        names.append(attrs.get("name", nd))
        etypes.append(attrs.get("entity_type", "Unknown"))
        comms.append(str(comm))
        colors.append(comm_id_to_color.get(comm, "#aaaaaa"))
        sizes.append(8 + 22 * (deg / max(max_degree, 1)))
        degs.append(deg)
        wdegs.append(float(attrs.get("weighted_degree", deg)))
        betw.append(round(float(attrs.get("betweenness_centrality", 0.0)), 5))
        clust.append(round(float(attrs.get("clustering_coefficient", 0.0)), 5))
        presence_list.append(attrs.get("corpus_presence", ""))
        texts_list.append(str(attrs.get("texts_present", "")))

    node_source = ColumnDataSource(dict(
        index           = node_ids,
        x               = xs,
        y               = ys,
        name            = names,
        entity_type     = etypes,
        community       = comms,
        fill_color      = colors,
        community_color = list(colors),
        size            = sizes,
        degree          = degs,
        weighted_degree = wdegs,
        betweenness     = betw,
        clustering      = clust,
        corpus_presence = presence_list,
        texts_present   = texts_list,
    ))

    # Edge data
    edge_xs, edge_ys, edge_weights_list, edge_alphas = [], [], [], []
    for u, v, d in G_f.edges(data=True):
        x0, y0 = pos.get(u, (0.0, 0.0))
        x1, y1 = pos.get(v, (0.0, 0.0))
        edge_xs.append([x0, x1])
        edge_ys.append([y0, y1])
        edge_weights_list.append(float(d.get("weight", 1)))
        edge_alphas.append(0.45)

    max_w       = max(edge_weights_list) if edge_weights_list else 1
    edge_widths = [0.5 + 4.5 * (w / max_w) for w in edge_weights_list]

    edge_source = ColumnDataSource(dict(
        xs      = edge_xs,
        ys      = edge_ys,
        weights = edge_weights_list,
        widths  = edge_widths,
        alpha   = edge_alphas,
    ))

    # Figure
    p = figure(
        title            = title,
        width            = width,
        height           = height,
        tools            = "pan,wheel_zoom,box_zoom,reset,tap",
        toolbar_location = "above",
        x_axis_type      = None,
        y_axis_type      = None,
    )
    p.title.text_font_size  = "13pt"
    p.background_fill_color = "#f8f8f8"
    p.grid.grid_line_color  = None

    # Edges
    p.multi_line(
        "xs", "ys",
        source     = edge_source,
        line_width = "widths",
        line_color = "#888888",
        line_alpha = "alpha",
    )

    # Nodes
    node_renderer = p.circle(
        "x", "y",
        source     = node_source,
        size       = "size",
        fill_color = "fill_color",
        line_color = "white",
        line_width = 1.5,
    )

    # Hover tooltips
    hover = HoverTool(
        renderers = [node_renderer],
        tooltips  = [
            ("Name",            "@name"),
            ("Q-ID",            "@index"),
            ("Type",            "@entity_type"),
            ("Community",       "@community"),
            ("Degree",          "@degree"),
            ("Weighted degree", "@weighted_degree"),
            ("Betweenness",     "@betweenness"),
            ("Clustering",      "@clustering"),
            ("Corpus presence", "@corpus_presence texts"),
            ("Texts",           "@texts_present"),
        ]
    )
    p.add_tools(hover)

    return p, node_source, edge_source

In [ ]:
# ── 5-C: Interactive plot with minimum-weight slider ──────────────────────────

def make_interactive_graph(G: nx.Graph,
                            title: str,
                            initial_min_weight: float = 1.0,
                            width: int = 950,
                            height: int = 720) -> None:
    """
    Renders an interactive Bokeh plot with hover tooltips and a slider
    for filtering edges by minimum weight (implemented via CustomJS).
    """
    p, node_source, edge_source = make_bokeh_graph(
        G, title=title, min_edge_weight=1.0,
        width=width, height=height
    )
    if p is None:
        return

    weights = edge_source.data["weights"]
    max_w   = int(max(weights)) if weights else 1

    slider = Slider(
        start=1, end=max(max_w, 2),
        value=int(initial_min_weight),
        step=1,
        title="Minimum edge weight"
    )

    callback = CustomJS(
        args=dict(edge_source=edge_source, slider=slider),
        code="""
            const weights   = edge_source.data['weights'];
            const threshold = slider.value;
            const new_alpha = [];
            for (let i = 0; i < weights.length; i++) {
                new_alpha.push(weights[i] >= threshold ? 0.45 : 0.0);
            }
            edge_source.data['alpha'] = new_alpha;
            edge_source.change.emit();
        """
    )
    slider.js_on_change("value", callback)

    show(column(p, slider))

In [ ]:
# ── 5-D: Corpus plot (colour toggle + two sliders) ───────────────────────────

def make_corpus_graph(G_corpus: nx.Graph,
                       corpus_nodes_full: pd.DataFrame,
                       title: str = "EcoCor Corpus Network",
                       width: int = 1100,
                       height: int = 820) -> None:
    """
    Interactive corpus graph with:
    - Slider: minimum edge weight
    - Slider: minimum corpus presence (nodes)
    - Select: colour nodes by community or corpus presence
    """
    # Set corpus-presence as a node attribute
    cp_map    = corpus_nodes_full.set_index("entity_qid")["corpus_presence"].to_dict()
    texts_map = corpus_nodes_full.set_index("entity_qid")["texts_present"].to_dict()
    nx.set_node_attributes(G_corpus, cp_map, "corpus_presence")
    nx.set_node_attributes(G_corpus, texts_map, "texts_present")

    p, node_source, edge_source = make_bokeh_graph(
        G_corpus, title=title, min_edge_weight=1.0,
        width=width, height=height
    )
    if p is None:
        return

    # Compute presence colours
    max_presence    = int(corpus_nodes_full["corpus_presence"].max())
    presence_colors = [
        presence_to_hex(int(cp) if cp != "" else 1, max_presence)
        for cp in node_source.data["corpus_presence"]
    ]
    node_source.data["presence_color"] = presence_colors

    # ── Slider: minimum edge weight ───────────────────────────────────────────
    weights  = edge_source.data["weights"]
    max_w    = int(max(weights)) if weights else 1
    slider_w = Slider(
        start=1, end=max(max_w, 2), value=1, step=1,
        title="Minimum edge weight"
    )
    cb_w = CustomJS(
        args=dict(edge_source=edge_source, slider=slider_w),
        code="""
            const weights   = edge_source.data['weights'];
            const threshold = slider.value;
            const new_alpha = [];
            for (let i = 0; i < weights.length; i++) {
                new_alpha.push(weights[i] >= threshold ? 0.45 : 0.0);
            }
            edge_source.data['alpha'] = new_alpha;
            edge_source.change.emit();
        """
    )
    slider_w.js_on_change("value", cb_w)

    # ── Slider: minimum corpus presence ──────────────────────────────────────
    slider_p = Slider(
        start=1, end=max(max_presence, 2), value=1, step=1,
        title="Minimum corpus presence (number of texts)"
    )
    cb_p = CustomJS(
        args=dict(node_source=node_source, slider=slider_p),
        code="""
            const presence  = node_source.data['corpus_presence'];
            const threshold = slider.value;
            const new_alpha = [];
            for (let i = 0; i < presence.length; i++) {
                new_alpha.push(parseInt(presence[i]) >= threshold ? 1.0 : 0.1);
            }
            node_source.data['line_alpha'] = new_alpha;
            node_source.change.emit();
        """
    )
    slider_p.js_on_change("value", cb_p)

    # ── Select: colouring mode ────────────────────────────────────────────────
    color_select = Select(
        title="Colour nodes by:",
        value="community",
        options=[("community", "Community"), ("corpus_presence", "Corpus presence")]
    )
    cb_color = CustomJS(
        args=dict(node_source=node_source, select=color_select),
        code="""
            const choice = select.value;
            const n      = node_source.data['fill_color'].length;
            const comm_c = node_source.data['community_color'];
            const pres_c = node_source.data['presence_color'];
            for (let i = 0; i < n; i++) {
                node_source.data['fill_color'][i] =
                    (choice === 'community') ? comm_c[i] : pres_c[i];
            }
            node_source.change.emit();
        """
    )
    color_select.js_on_change("value", cb_color)

    show(column(p, row(color_select, slider_w, slider_p)))

### 5-E: Visualise a Single Text Network

Adjust `TEXT_TO_VISUALIZE` to one of the text IDs listed above.

In [ ]:
# ── List available text IDs ───────────────────────────────────────────────────
for tid in per_text_results:
    meta = text_meta_lookup.get(tid, {})
    G    = per_text_results[tid]["G"]
    print(f"  {tid}  –  {meta.get('title', '')}  "
          f"({G.number_of_nodes()} nodes)")

In [ ]:
# TEXT_TO_VISUALIZE anpassen:
TEXT_TO_VISUALIZE = list(per_text_results.keys())[0]   # erster Text als Standard

result = per_text_results[TEXT_TO_VISUALIZE]
meta   = text_meta_lookup.get(TEXT_TO_VISUALIZE, {})

make_interactive_graph(
    result["G"],
    title=f"EcoCor: {meta.get('title', TEXT_TO_VISUALIZE)}  (window={WINDOW_SIZE})",
    initial_min_weight=1.0,
)

### 5-F: Visualise the Corpus Network

In [ ]:
make_corpus_graph(
    G_corpus,
    corpus_nodes_full,
    title=f"EcoCor Corpus Network – '{TARGET_CORPUS.upper()}'",
)

## Section 6 – Summary Statistics

In [ ]:
summary_rows = []

for text_id, result in per_text_results.items():
    G    = result["G"]
    meta = result["text_meta"]
    if G.number_of_nodes() == 0:
        continue

    if G.number_of_edges() > 0:
        comms     = list(nx_community.greedy_modularity_communities(G, weight="weight"))
        mod_score = nx_community.modularity(G, comms, weight="weight")
        avg_clust = nx.average_clustering(G, weight="weight")
    else:
        comms, mod_score, avg_clust = [], 0.0, 0.0

    summary_rows.append({
        "text_id":         text_id,
        "title":           meta.get("title", ""),
        "nodes":           G.number_of_nodes(),
        "edges":           G.number_of_edges(),
        "communities":     len(comms),
        "modularity":      round(mod_score, 4),
        "avg_degree":      round(2 * G.number_of_edges() / max(G.number_of_nodes(), 1), 2),
        "density":         round(nx.density(G), 4),
        "avg_clustering":  round(avg_clust, 4),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(
    OUT_DIR / "corpus" / f"{TARGET_CORPUS}_corpus_summary.csv",
    index=False, encoding="utf-8"
)

display(summary_df.sort_values("nodes", ascending=False))

In [ ]:
# Corpus-level statistics
print("Corpus network statistics:")
print(f"  Nodes           : {G_corpus.number_of_nodes()}")
print(f"  Edges           : {G_corpus.number_of_edges()}")
print(f"  Density         : {nx.density(G_corpus):.4f}")
if G_corpus.number_of_edges() > 0:
    comms_corp  = list(nx_community.greedy_modularity_communities(G_corpus, weight="weight"))
    mod_corp    = nx_community.modularity(G_corpus, comms_corp, weight="weight")
    avg_clust_c = nx.average_clustering(G_corpus, weight="weight")
    print(f"  Communities     : {len(comms_corp)}")
    print(f"  Modularity      : {mod_corp:.4f}")
    print(f"  Avg. clustering : {avg_clust_c:.4f}")
print(f"\nTexts in corpus : {len(per_text_results)}")
print(f"Entity types    : {ENTITY_TYPES}")
print(f"Window size     : {WINDOW_SIZE} paragraph(s)")